# AETHER — Stage 7: Real-Scale Depth Transformer Voice Head

A real training run, not a structural probe: ~10,000 diverse English phrases (not 20-24),
a real Depth Transformer (codebook `k` conditioned on codebooks `0..k-1` of the same frame --
this is what fixes the "words dropping out" problem from Stage 5), and a model an order of
magnitude bigger (8 layers, d_model=512 vs. Stage 5's 2 layers / 128).

**Everything is saved to Google Drive**, not ephemeral `/content` -- the data cache,
checkpoints, and the loss log. This is built for a multi-day run with real interruption
risk: if Colab disconnects, just re-run the same cells and both stages (data generation
and training) resume from exactly where they left off.

**Two separate steps, can run in different Colab sessions:**
1. Data generation (hidden state + teacher tokens per phrase) -- can take hours.
2. Training (checkpointed, resumable) -- can take days.

**Watching progress:** both steps print a progress line to the cell output every batch/log
interval (elapsed time, throughput, ETA) -- you should not need to guess whether a multi-day
run is stuck. The loss log and report.json on Drive are also readable live from another
Colab cell or from your own machine while training is still running.

In [ ]:
REPO_URL = "https://github.com/Manifestro/aether.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}
PHRASE_COUNT = "10000"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/aether-stage7"  # @param {type:"string"}


## Mount Google Drive

Do this before anything else -- without it, the cache and checkpoints are lost the moment
this Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive root:", DRIVE_ROOT)


In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/aether")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml,audio]"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "torchvision", "torchaudio"],
    check=True,
)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(tests.stdout)
if tests.returncode != 0:
    raise RuntimeError("Tests failed")


## Step 1 -- data generation (resumable)

Safe to interrupt and re-run this cell as many times as needed -- phrases already cached
(by `phrase_id`) on Drive are not regenerated, only the remaining ones are processed.
Progress prints every batch: how many done, throughput (phrases/sec), elapsed time, and an
ETA for the rest of this run. This step streams its output live -- you don't need to wait
for it to finish to see progress.

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
data_output = Path(DRIVE_ROOT) / "data_pipeline_artifacts"
cache_path = Path(DRIVE_ROOT) / "cache.jsonl"

command = [
    sys.executable, "-u", "-m", "aether.experiments.colab_stage7_data_pipeline",
    "--allow-download",
    "--model", MODEL_ID,
    "--phrase-count", PHRASE_COUNT,
    "--cache-path", str(cache_path),
    "--output-dir", str(data_output),
]
# Streamed (not captured-then-printed) so progress is visible live, cell by cell,
# during what can be an hours-long run -- important for "is this actually still going".
process = subprocess.Popen(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for line in process.stdout:
    print(line, end="")
exit_code = process.wait()
print("\nExit code:", exit_code)


In [ ]:
import json

report = json.loads((data_output / "report.json").read_text(encoding="utf-8"))
print("Status:", report.get("status"))
print("Already cached at start:", report.get("already_cached_at_start"))
print("Remaining at start:", report.get("remaining_at_start"))
print("Processed this run:", report.get("processed_this_run"))
print("Remaining now:", report.get("remaining_now"))


## Step 2 -- training (checkpointed, resumable)

Also safe to interrupt/re-run -- the Drive checkpoint holds the model, optimizer, epoch,
and step count, so re-running this exact command continues instead of restarting. Progress
prints every `log_every_steps` steps: loss, steps/sec, elapsed, ETA to the configured epoch
count -- this is the number to watch over a multi-day run.

In [ ]:
train_output = Path(DRIVE_ROOT) / "train_artifacts"
checkpoint_path = Path(DRIVE_ROOT) / "checkpoint.pt"
loss_log_path = Path(DRIVE_ROOT) / "loss_log.jsonl"

command = [
    sys.executable, "-u", "-m", "aether.experiments.colab_stage7_train",
    "--cache-path", str(cache_path),
    "--checkpoint-path", str(checkpoint_path),
    "--loss-log-path", str(loss_log_path),
    "--output-dir", str(train_output),
]
process = subprocess.Popen(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for line in process.stdout:
    print(line, end="")
exit_code = process.wait()
print("\nExit code:", exit_code)


In [ ]:
report = json.loads((train_output / "report.json").read_text(encoding="utf-8"))
print("Status:", report.get("status"))
print("Resumed from existing checkpoint:", report.get("resumed_from_existing_checkpoint"))
print("Train/val count:", report.get("train_count"), "/", report.get("val_count"))
print("Result:", json.dumps(report.get("result"), indent=2))

if loss_log_path.exists():
    lines = loss_log_path.read_text(encoding="utf-8").strip().splitlines()
    print(f"\n{len(lines)} loss-log entries; last 10:")
    for line in lines[-10:]:
        print(line)


## Check progress from a fresh cell, any time (even mid-run, from another session)

Run this cell any time you want a status update without touching the running process --
it just reads the Drive files, so it works from a separate Colab session too, or from your
own machine if you mount the same Drive folder locally.

In [ ]:
import json
from pathlib import Path

drive_root = Path(DRIVE_ROOT)
for name, path in [
    ("data pipeline report", drive_root / "data_pipeline_artifacts" / "report.json"),
    ("training report", drive_root / "train_artifacts" / "report.json"),
]:
    if path.exists():
        report = json.loads(path.read_text(encoding="utf-8"))
        print(f"=== {name} ===")
        print("status:", report.get("status"), "| finished_at:", report.get("finished_at"))
    else:
        print(f"=== {name} === not started yet")

loss_log = drive_root / "loss_log.jsonl"
if loss_log.exists():
    lines = loss_log.read_text(encoding="utf-8").strip().splitlines()
    print(f"\nloss_log.jsonl: {len(lines)} entries, last 5:")
    for line in lines[-5:]:
        print(line)


## If something fails

`report.json` (under `data_pipeline_artifacts/` or `train_artifacts/` on Drive) is written
at every step, and `traceback.txt` next to it shows exactly where. Since everything lives on
Drive, nothing is lost between Colab sessions -- just reopen this notebook and re-run the
same cells.